In [1]:
import networkx as nx
from bp.world import random_grid_world, Scenario

In [2]:
world = random_grid_world(
    rows=4,
    cols=4,
    demands={
        # ((0, 0), (3, 3)): 10,
        ((1, 0), (1, 3)): 5,
    },
    seed=0,
)
G = world.network.graph
nominal = Scenario.from_world("nominal", world)

print(f"{world.total_population=}")
print(f"{world.total_demand((0, 0), (3, 3))=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")

world.total_population=5
world.total_demand((0, 0), (3, 3))=0
world.total_demand((1, 0), (1, 3))=5


In [3]:
from itertools import pairwise
def edge_path(path):
    """Given a path consisting of vertices, return the corresponding edge-path"""
    return list(pairwise(path)) # list for state safety, can remove if careful though

In [4]:
import numpy as np

def debug_scenario(volumes):
    # (volume - 1) b/c need to recalculate b/c of off-by-one-error
    active_travel_time = world.network.actual_travel_times({a: volume - 1 for a, volume in volumes.items()})
    active_arcs = {a: t for a, t in active_travel_time.items() if volumes[a] > 0}

    travel_time = sum(t * volumes[a] for a, t in active_arcs.items())
    average_travel_time = travel_time / world.total_population
    print(f"{travel_time=:.2f}")
    print(f"{average_travel_time=:.2f}")

    nominal_arc_time = sum(nominal.travel_time[a] for a in active_arcs)
    realized_arc_time = sum(active_arcs.values())
    average_arc_time = realized_arc_time / len(active_arcs)
    arc_time_increase = realized_arc_time - nominal_arc_time
    average_arc_time_increase = arc_time_increase / len(active_arcs)
    print(f"{average_arc_time=:.2f} (along arcs w/ non-zero flow/volume)")
    print(f"{average_arc_time_increase=:.2f} (+{arc_time_increase / nominal_arc_time * 100:.2f}%, overall, not per-arc)")

    flow = np.array([volumes.get(arc, 0) for arc in world.ordered_arcs], dtype=float)
    capacity_used = flow / world.network._arc_array("capacity") * 100
    active_capacity_used = capacity_used[capacity_used > 0]
    average_capacity_used = np.average(active_capacity_used)
    most_capacity_used = np.max(capacity_used)
    print(f"{most_capacity_used=:.2f}%")
    print(f"{average_capacity_used=:.2f}%")

    # should sort then bin-search for `congested` b/c sorting but copy paste lazy
    congested = sum(active_arcs[a] > nominal.travel_time[a] + 1e-9 for a in active_arcs)
    print(f"arcs whose travel time rose under congestion: {congested}/{len(world.A)}")

    most_congested = sorted(((a, (active_arcs[a] - nominal.travel_time[a]) / nominal.travel_time[a]) for a in active_arcs), reverse=True, key=lambda x: x[1])
    for i, ((orig, dest), delta) in enumerate(most_congested[:congested], 1):
        print(f"{i}: {orig}->{dest}: +{delta * 100:.2f}%")

    return travel_time

In [5]:
routes = {
    plr: edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight="travel_time"))
    for plr in world.individuals
}

volumes = {arc: 0 for arc in world.A}
for route in routes.values():
    for arc in route:
        volumes[arc] += 1

# realized_concurrent = world.realize(routes)
print("CONCURRENT:")
debug_scenario(volumes)

CONCURRENT:
travel_time=121.37
average_travel_time=24.27
average_arc_time=8.09 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=1.44 (+21.59%, overall, not per-arc)
most_capacity_used=163.88%
average_capacity_used=123.79%
arcs whose travel time rose under congestion: 3/48
1: (1, 2)->(1, 3): +44.31%
2: (1, 0)->(1, 1): +7.43%
3: (1, 1)->(1, 2): +6.82%


121.37450293898158

In [6]:
def dynamic_weight(orig, dest, _):
    return current_travel_times[(orig, dest)]

volumes = {arc: 0 for arc in world.A}
routes = {}
for plr in world.individuals:
    current_travel_times = world.network.actual_travel_times(volumes)
    route = edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight=dynamic_weight))
    routes[plr] = route
    for arc in route:
        volumes[arc] += 1

# realized_sequential = world.realize(routes)
print("SEQUENTIAL:")
debug_scenario(volumes)

SEQUENTIAL:
travel_time=121.37
average_travel_time=24.27
average_arc_time=8.09 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=1.44 (+21.59%, overall, not per-arc)
most_capacity_used=163.88%
average_capacity_used=123.79%
arcs whose travel time rose under congestion: 3/48
1: (1, 2)->(1, 3): +44.31%
2: (1, 0)->(1, 1): +7.43%
3: (1, 1)->(1, 2): +6.82%


121.37450293898158